# ACC102 Track4: WRDS Financial Data Analysis
Data Source: WRDS Compustat
5 Companies: AAPL, MSFT, GOOGL, AMZN, NVDA
Period: 2010–2024

In [ ]:
# 1. Import libraries
import wrds
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 2. Connect to WRDS
db = wrds.Connection(wrds_username="YOUR_WRDS_USERNAME")

In [ ]:
# 3. Extract data from WRDS Compustat
query = """
    SELECT tic, conm, datadate, ni, sale, at, lt, roe
    FROM comp.funda
    WHERE tic IN ('AAPL','MSFT','GOOGL','AMZN','NVDA')
    AND EXTRACT(YEAR FROM datadate) BETWEEN 2010 AND 2024
    AND datafmt='STD' AND consol='C' AND indfmt='INDL'
    ORDER BY tic, datadate;
"""
df = db.raw_sql(query)
db.close()

In [ ]:
# 4. Data cleaning & calculation
df["year"] = pd.to_datetime(df["datadate"]).dt.year
df = df.dropna(subset=["at","sale"])
df["roa"] = df["ni"] / df["at"]
df["leverage"] = df["lt"] / df["at"]
df["profit_margin"] = df["ni"] / df["sale"]

In [ ]:
# 5. Show data
print(df.head())
print(df.describe())

In [ ]:
# 6. Visualization
plt.figure(figsize=(10, 6))
for tic in df["tic"].unique():
    sub = df[df["tic"] == tic]
    plt.plot(sub["year"], sub["roe"], marker='o', label=tic)
plt.title("ROE Trend 2010–2024")
plt.xlabel("Year")
plt.ylabel("ROE")
plt.legend()
plt.grid(True)
plt.show()